# 1. Train YOLO Models

In [1]:
!nvidia-smi

Tue Mar 24 08:45:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 2.&nbsp;Upload Image Dataset and Prepare Training Data

## 2.1 Upload images


In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

!cp /content/gdrive/MyDrive/yolo/data.zip /content

Mounted at /content/gdrive


## 2.2 Split images into train and validation folders

In [3]:
!unzip -q /content/data.zip -d /content/custom_data

In [8]:
from pathlib import Path
import random
import os
import sys
import shutil

data_path = "/content/custom_data"
train_percent = 0.3

if not os.path.isdir(data_path):
    print(f'Directory specified by data_path ({data_path}) not found. Verify the path is correct.')
elif train_percent < .01 or train_percent > 0.99:
    print('Invalid entry for train_percent. Please enter a number between .01 and .99.')
else:
    val_percent = 1 - train_percent

    input_image_path = os.path.join(data_path, 'images')
    input_label_path = os.path.join(data_path, 'labels')

    cwd = os.getcwd()
    train_img_path = os.path.join(cwd, 'data/train/images')
    train_txt_path = os.path.join(cwd, 'data/train/labels')
    val_img_path = os.path.join(cwd, 'data/validation/images')
    val_txt_path = os.path.join(cwd, 'data/validation/labels')

    for dir_path in [train_img_path, train_txt_path, val_img_path, val_txt_path]:
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)
            print(f'Created folder at {dir_path}.')

    img_file_list = [path for path in Path(input_image_path).rglob('*') if path.is_file()]
    txt_file_list = [path for path in Path(input_label_path).rglob('*') if path.is_file()]

    print(f'Number of image files: {len(img_file_list)}')
    print(f'Number of annotation files: {len(txt_file_list)}')

    file_num = len(img_file_list)
    train_num = int(file_num * train_percent)
    val_num = file_num - train_num
    print("Images moving to train: %d" % train_num)
    print("Images moving to validation: %d" % val_num)

    for i, set_num in enumerate([train_num, val_num]):
        for ii in range(set_num):
            if not img_file_list:
                break
            img_path = random.choice(img_file_list)
            img_fn = img_path.name
            base_fn = img_path.stem
            txt_fn = base_fn + '.txt'
            txt_path = os.path.join(input_label_path, txt_fn)

            if i == 0:
                new_img_path, new_txt_path = train_img_path, train_txt_path
            elif i == 1:
                new_img_path, new_txt_path = val_img_path, val_txt_path

            shutil.copy(img_path, os.path.join(new_img_path, img_fn))
            if os.path.exists(txt_path):
                shutil.copy(txt_path, os.path.join(new_txt_path, txt_fn))

            img_file_list.remove(img_path)

Number of image files: 1086
Number of annotation files: 1086
Images moving to train: 325
Images moving to validation: 761


# 3.&nbsp;Install Requirements


In [5]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.1 MB/s eta 0:00:00


# 4.&nbsp;Configure Training


In [ ]:
import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

path_to_classes_txt = '/content/custom_data/classes.json'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
os.system(f'cat {path_to_data_yaml}')

Created config file at /content/data.yaml

File contents:

path: /content/data
train: train/images
val: validation/images
nc: 4
names:
- ARJUNA
- CAT
- LION
- NANDHI


# 5.&nbsp;Train Model

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs=60 imgsz=640

#6.&nbsp;Test Model

In [ ]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=data/validation/images save=True

In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'/content/runs/detect/predict/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')

#7.&nbsp;Deploy Model

## 7.1 Download YOLO Model


In [ ]:
!mkdir /content/my_model
!cp /content/runs/detect/train/weights/best.pt /content/my_model/my_model.pt
!cp -r /content/runs/detect/train /content/my_model

%cd my_model
!zip /content/my_model.zip my_model.pt
!zip -r /content/my_model.zip train
%cd /content

In [ ]:
from google.colab import files

files.download('/content/my_model.zip')